In [12]:
import pandas as pd
import numpy as np
import os

RAW = '../data/raw/'
PROCESSED = '../data/processed/'
FINAL = '../data/final/'

In [13]:
#Cargamos los dos archivos de alquiler
df_barrio_raw = pd.read_excel(RAW + 'alquiler_barrio.xlsx', header=None)
df_distrito_raw = pd.read_excel(RAW + 'alquiler_distrito.xlsx', header=None)

print('ALQUILER POR BARRIO')
print(f'Dimensiones: {df_barrio_raw.shape}')
print(df_barrio_raw.iloc[:8, :5].to_string())

print('\nALQUILER POR DISTRITO')
print(f'Dimensiones: {df_distrito_raw.shape}')
print(df_distrito_raw.iloc[:8, :5].to_string())

C:\Users\raned\anaconda3\envs\environment_uoc20232pec3-actual\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


ALQUILER POR BARRIO
Dimensiones: (621, 15)
                                                                                        0                       1                 2      3        4
0                                                               E. Edificación y Vivienda                     NaN               NaN    NaN      NaN
1                                                                  Mercado de la vivienda                     NaN               NaN    NaN      NaN
2                                                                  Precios de la vivienda                     NaN               NaN    NaN      NaN
3  4.3.21.D. Evolución del precio de alquiler de la vivienda (€/m2) por Distrito y Barrio                     NaN               NaN    NaN      NaN
4                                                                                     NaN                     NaN               NaN  Enero  Febrero
5                                                                    

C:\Users\raned\anaconda3\envs\environment_uoc20232pec3-actual\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [14]:
#Ver meses disponibles y años en ambos archivos
print('ALQUILER POR BARRIO')
meses_barrio = [v for v in df_barrio_raw.iloc[4, :].tolist() if v is not None and str(v) != 'nan']
print(f'Meses: {meses_barrio}')

años_barrio = sorted(set(
    str(row) for row in df_barrio_raw.iloc[5:, 0]
    if str(row).strip().isdigit()))
print(f'Años: {años_barrio}')

print('\nALQUILER POR DISTRITO')
meses_distrito = [v for v in df_distrito_raw.iloc[4, :].tolist() if v is not None and str(v) != 'nan']
print(f'Meses: {meses_distrito}')

años_distrito = sorted(set(
    str(row) for row in df_distrito_raw.iloc[5:, 0]
    if str(row).strip().isdigit()))
print(f'Años: {años_distrito}')

#Revisar cuántas unidades geográficas hay en cada archivo
geo_barrio = set(
    str(row) for row in df_barrio_raw.iloc[5:, 2]
    if row is not None and str(row) != 'nan')
geo_distrito = set(
    str(row) for row in df_distrito_raw.iloc[5:, 1]
    if row is not None and str(row) != 'nan')
print(f'\nUnidades geográficas barrio: {len(geo_barrio)}')
print(f'Unidades geográficas distrito: {len(geo_distrito)}')

ALQUILER POR BARRIO
Meses: ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
Años: ['2023', '2024', '2025', '2026']

ALQUILER POR DISTRITO
Meses: ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
Años: ['2018', '2019', '2020', '2021', '2022', '2023', '2024']

Unidades geográficas barrio: 153
Unidades geográficas distrito: 22


In [15]:
#Corregimos función para manejar el guión como valor nulo
def limpiar_decimal(valor):
    if pd.isna(valor):
        return np.nan
    if str(valor).strip() == '-':
        return np.nan
    try:
        return float(str(valor).replace(',', '.').strip())
    except:
        return np.nan

registros_barrio = []
for idx, row in df_barrio_raw.iterrows():
    año = str(row[0]).strip()
    if año != '2023':
        continue
    
    barrio = str(row[2]).strip()
    codigo = barrio.split('.')[0].strip()
    if not (len(codigo) == 3 and codigo.isdigit()):
        continue
    
    # Calcular media anual de los 12 meses
    valores_mensuales = [limpiar_decimal(row[i]) for i in range(3, 15)]
    valores_validos = [v for v in valores_mensuales if not np.isnan(v)]
    
    if len(valores_validos) > 0:
        media_anual = round(sum(valores_validos) / len(valores_validos), 2)
    else:
        media_anual = np.nan
    
    registros_barrio.append({
        'cod_barrio': codigo,
        'barrio': barrio,
        'alquiler_2023': media_anual,
        'meses_disponibles': len(valores_validos)})

df_alq_barrio = pd.DataFrame(registros_barrio)
print(f'Barrios con precio alquiler 2023: {len(df_alq_barrio)}')
print(f'Valores nulos: {df_alq_barrio.alquiler_2023.isnull().sum()}')
print(f'\nMeses disponibles por barrio:')
print(df_alq_barrio.meses_disponibles.value_counts().to_string())
print(f'\nMuestra:')
print(df_alq_barrio.head(10).to_string())

Barrios con precio alquiler 2023: 131
Valores nulos: 12

Meses disponibles por barrio:
meses_disponibles
12    81
0     12
5      7
3      7
11     5
7      5
6      4
9      2
1      2
10     2
2      2
4      1
8      1

Muestra:
  cod_barrio            barrio  alquiler_2023  meses_disponibles
0        011      011. Palacio          19.87                 12
1        012  012. Embajadores          19.79                 12
2        013       013. Cortes          20.39                 12
3        014     014. Justicia          22.60                 12
4        015  015. Universidad          21.07                 12
5        016          016. Sol          20.49                 12
6        021     021. Imperial          16.65                 12
7        022      022. Acacias          17.18                 12
8        023      023. Chopera          16.29                 12
9        024      024. Legazpi          15.73                 12


In [16]:
#Revisar qué barrios tienen datos incompletos
print('Barrios con menos de 12 meses de datos:')
print(df_alq_barrio[df_alq_barrio['meses_disponibles'] < 12][
    ['cod_barrio', 'barrio', 'alquiler_2023', 'meses_disponibles']
].sort_values('meses_disponibles').to_string())

Barrios con menos de 12 meses de datos:
    cod_barrio                             barrio  alquiler_2023  meses_disponibles
12         027                        027. Atocha            NaN                  0
127        212                    212. Aeropuerto            NaN                  0
120        203                       203. Amposta            NaN                  0
119        202                        202. Hellín            NaN                  0
43         081                      081. El Pardo            NaN                  0
112        182                 182. Santa Eugenia            NaN                  0
107        172                 172. San Cristóbal            NaN                  0
98         158                       158. Atalaya            NaN                  0
86         142                       142. Horcajo            NaN                  0
85         141                       141. Pavones            NaN                  0
63         106                106. C

In [17]:
#Imputamos nulos con la media del distrito
#El código de distrito son los primeros 2 dígitos del código de barrio
df_alq_barrio['cod_distrito'] = df_alq_barrio['cod_barrio'].str[:2]

#Calcular media por distrito (solo barrios con datos)
media_distrito = df_alq_barrio.groupby('cod_distrito')['alquiler_2023'].mean().round(2)
print('Media de alquiler por distrito:')
print(media_distrito.to_string())

#Vemos qué distrito tienen los barrios nulos
print('\nBarrios nulos y su distrito:')
nulos = df_alq_barrio[df_alq_barrio['alquiler_2023'].isna()]
for _, row in nulos.iterrows():
    dist = row['cod_distrito']
    media = media_distrito.get(dist, np.nan)
    print(f'  {row["barrio"]} (dist. {dist}) → imputar con {media}')

Media de alquiler por distrito:
cod_distrito
01    20.70
02    16.68
03    17.22
04    20.84
05    17.86
06    17.26
07    20.18
08    13.43
09    13.72
10    13.10
11    13.47
12    13.51
13    13.10
14    12.13
15    15.03
16    13.98
17    12.15
18    12.44
19    12.00
20    12.87
21    12.70

Barrios nulos y su distrito:
  027. Atocha (dist. 02) → imputar con 16.68
  081. El Pardo (dist. 08) → imputar con 13.43
  106. Cuatro Vientos (dist. 10) → imputar con 13.1
  121. Orcasitas (dist. 12) → imputar con 13.51
  141. Pavones (dist. 14) → imputar con 12.13
  142. Horcajo (dist. 14) → imputar con 12.13
  158. Atalaya (dist. 15) → imputar con 15.03
  172. San Cristóbal (dist. 17) → imputar con 12.15
  182. Santa Eugenia (dist. 18) → imputar con 12.44
  202. Hellín (dist. 20) → imputar con 12.87
  203. Amposta (dist. 20) → imputar con 12.87
  212. Aeropuerto (dist. 21) → imputar con 12.7


In [18]:
#Aplicamos imputación con media del distrito
df_alq_barrio['alquiler_2023'] = df_alq_barrio.apply(
    lambda row: media_distrito[row['cod_distrito']] 
    if pd.isna(row['alquiler_2023']) 
    else row['alquiler_2023'],
    axis=1).round(2)

print(f'Valores nulos tras imputación: {df_alq_barrio.alquiler_2023.isnull().sum()}')
print(f'\nEstadísticas precio alquiler 2023:')
print(df_alq_barrio['alquiler_2023'].describe().round(2))

print(f'\nTop 10 barrios más caros:')
print(df_alq_barrio[['barrio', 'alquiler_2023']].sort_values('alquiler_2023', ascending=False).head(10).to_string())

print(f'\nTop 10 barrios más baratos:')
print(df_alq_barrio[['barrio', 'alquiler_2023']].sort_values('alquiler_2023', ascending=True).head(10).to_string())

Valores nulos tras imputación: 0

Estadísticas precio alquiler 2023:
count    131.00
mean      15.02
std        3.24
min        0.00
25%       12.70
50%       14.12
75%       16.87
max       24.20
Name: alquiler_2023, dtype: float64

Top 10 barrios más caros:
              barrio  alquiler_2023
19    041. Recoletos          24.20
3      014. Justicia          22.60
24   046. Castellana          22.38
39    073. Trafalgar          22.09
40      074. Almagro          21.92
20         042. Goya          21.20
23        045. Lista          21.10
4   015. Universidad          21.07
5           016. Sol          20.49
2        013. Cortes          20.39

Top 10 barrios más baratos:
                  barrio  alquiler_2023
56       096. El Plantío           0.00
44   082. Fuentelarreina          10.91
116     193. Valderribas          10.97
110         175. Ángeles          11.50
74       123. San Fermín          11.51
117    194. El Cañaveral          11.55
79        131. Entrevías          1

In [19]:
#Revisamos El Plantío
print('Detalle El Plantío:')
print(df_alq_barrio[df_alq_barrio['cod_barrio'] == '096'].to_string())
print('\nValor original')
for idx, row in df_barrio_raw.iterrows():
    if str(row[0]).strip() == '2023' and '096' in str(row[2]):
        print(row.tolist())

Detalle El Plantío:
   cod_barrio           barrio  alquiler_2023  meses_disponibles cod_distrito
56        096  096. El Plantío            0.0                 12           09

Valor original
['2023', '09. Moncloa-Aravaca', '096. El Plantío', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0']


In [20]:
#Sustituimos 0 por media del distrito 09
media_dist_09 = media_distrito['09']
print(f'Media distrito 09 (Moncloa-Aravaca): {media_dist_09}')

df_alq_barrio.loc[df_alq_barrio['cod_barrio'] == '096', 'alquiler_2023'] = media_dist_09

print(f'\nValor corregido El Plantío: {df_alq_barrio[df_alq_barrio["cod_barrio"] == "096"]["alquiler_2023"].values[0]}')
print(f'\nNuevo mínimo de alquiler:')
print(df_alq_barrio[['barrio', 'alquiler_2023']]
      .sort_values('alquiler_2023')
      .head(5).to_string())

Media distrito 09 (Moncloa-Aravaca): 13.72

Valor corregido El Plantío: 13.72

Nuevo mínimo de alquiler:
                  barrio  alquiler_2023
44   082. Fuentelarreina          10.91
116     193. Valderribas          10.97
110         175. Ángeles          11.50
74       123. San Fermín          11.51
117    194. El Cañaveral          11.55


In [21]:
#ALQUILER POR DISTRITO. Variación 2018-2023

registros_distrito = []
for idx, row in df_distrito_raw.iterrows():
    año = str(row[0]).strip()
    if año not in ['2018', '2023']:
        continue
    
    distrito = str(row[1]).strip()
    #Excluimos total ciudad
    codigo_dist = distrito.split('.')[0].strip()
    if not (len(codigo_dist) == 2 and codigo_dist.isdigit()):
        continue
    
    #Media anual de los 12 meses
    valores_mensuales = [limpiar_decimal(row[i]) for i in range(2, 14)]
    valores_validos = [v for v in valores_mensuales if not np.isnan(v)]
    
    if len(valores_validos) > 0:
        media_anual = round(sum(valores_validos) / len(valores_validos), 2)
    else:
        media_anual = np.nan
    
    registros_distrito.append({
        'cod_distrito': codigo_dist,
        'distrito': distrito,
        'año': int(año),
        'alquiler_medio': media_anual
    })

df_dist = pd.DataFrame(registros_distrito)

#Pivotamos para tener 2018 y 2023 en columnas separadas
df_dist_pivot = df_dist.pivot_table(
    index=['cod_distrito', 'distrito'],
    columns='año',
    values='alquiler_medio').reset_index()

df_dist_pivot.columns = ['cod_distrito', 'distrito', 'alquiler_dist_2018', 'alquiler_dist_2023']

#Calculamos variación porcentual
df_dist_pivot['var_alquiler_dist_1823'] = (
    (df_dist_pivot['alquiler_dist_2023'] - df_dist_pivot['alquiler_dist_2018']) 
    / df_dist_pivot['alquiler_dist_2018'] * 100).round(2)

print(f'Distritos procesados: {len(df_dist_pivot)}')
print('\nVariación alquiler 2018-2023 por distrito:')
print(df_dist_pivot[['distrito', 'alquiler_dist_2018', 'alquiler_dist_2023', 'var_alquiler_dist_1823']].sort_values('var_alquiler_dist_1823', ascending=False).to_string())

Distritos procesados: 21

Variación alquiler 2018-2023 por distrito:
                   distrito  alquiler_dist_2018  alquiler_dist_2023  var_alquiler_dist_1823
16           17. Villaverde                9.75               12.35                   26.67
11                12. Usera               10.77               13.56                   25.91
9                10. Latina               10.96               13.53                   23.45
12   13. Puente de Vallecas               11.05               13.63                   23.35
13            14. Moratalaz               10.21               12.58                   23.21
10          11. Carabanchel               11.12               13.40                   20.50
18            19. Vicálvaro                9.76               11.72                   20.08
17    18. Villa de Vallecas               10.66               12.61                   18.29
8       09. Moncloa-Aravaca               14.19               16.48                   16.14
6          

In [22]:
#Asignamos la variación del distrito a cada barrio
df_alq_barrio['cod_distrito'] = df_alq_barrio['cod_barrio'].str[:2]

df_alq_final = df_alq_barrio.merge(
    df_dist_pivot[['cod_distrito', 'var_alquiler_dist_1823']],
    on='cod_distrito',
    how='left')

#Seleccionamos las columnas finales
df_alquiler_final = df_alq_final[[
    'cod_barrio',
    'barrio',
    'alquiler_2023',
    'var_alquiler_dist_1823']].copy()


print(f'Dimensiones: {df_alquiler_final.shape}')
print(f'Valores nulos:\n{df_alquiler_final.isnull().sum()}')
print(f'\nMuestra:')
print(df_alquiler_final.head(10).to_string())

df_alquiler_final.to_csv(PROCESSED + 'df_alquiler.csv', index=False)

Dimensiones: (131, 4)
Valores nulos:
cod_barrio                0
barrio                    0
alquiler_2023             0
var_alquiler_dist_1823    0
dtype: int64

Muestra:
  cod_barrio            barrio  alquiler_2023  var_alquiler_dist_1823
0        011      011. Palacio          19.87                   11.73
1        012  012. Embajadores          19.79                   11.73
2        013       013. Cortes          20.39                   11.73
3        014     014. Justicia          22.60                   11.73
4        015  015. Universidad          21.07                   11.73
5        016          016. Sol          20.49                   11.73
6        021     021. Imperial          16.65                   14.04
7        022      022. Acacias          17.18                   14.04
8        023      023. Chopera          16.29                   14.04
9        024      024. Legazpi          15.73                   14.04
